# Walkthrough: spreadsheet → map
Run one cell at a time (Shift+Enter) and **look at every output**. Follow along with LEARN.md.

In [ ]:
import os
os.chdir('..')   # work from the project folder, where data/ lives
import pandas as pd
from election_map.parse import read_poll_by_poll, parse_sheet
from election_map.metrics import summarize

## Step 0: look at the raw sheet
`header=None` shows the file exactly as it is. Find the two blocks (city wards 1 and 7).

In [ ]:
raw = pd.read_excel('data/Poll_By_Poll_TDSB.xlsx', sheet_name='Ward 1', header=None)
raw.iloc[:, :6]

## Step 1: parse into long format

In [ ]:
long = read_poll_by_poll('data/Poll_By_Poll_Mayor.xlsx')
print(len(long), 'rows')
long.head()

In [ ]:
long.groupby('candidate').votes.sum().sort_values(ascending=False).head()

### Exercise 2: break it on purpose
Change one number and watch the totals check catch it.

In [ ]:
raw = pd.read_excel('data/Poll_By_Poll_Mayor.xlsx', sheet_name='Ward 1', header=None)
raw.iat[5, 3] = 999
try:
    parse_sheet(raw, 'Ward 1')
except Exception as e:
    print(type(e).__name__, '→', e)

## Step 2: one row per subdivision

In [ ]:
s = summarize(long, compare=('Chow Olivia', 'Bradford Brad'))
s.head()

In [ ]:
s.winner.value_counts()

### Exercise 3: the 10 closest subdivisions

In [ ]:
s.assign(gap=s.a_minus_b.abs()).sort_values('gap').head(10)

## Step 3: boundaries and the join
This uses your real file if `data/subdivisions.gpkg` exists (exported from QGIS); otherwise it uses **fake squares** so you can still practise.

In [ ]:
from election_map.geo import load_boundaries, join
if os.path.exists('data/subdivisions.gpkg'):
    shapes = load_boundaries('data/subdivisions.gpkg')
else:
    from tests.fake_boundaries import fake_boundaries
    shapes = fake_boundaries(s)
    print('Using FAKE squares, not real geography')
shapes.head()

In [ ]:
gdf, report = join(shapes, s)
report

## Step 4: the map
`make_map()` saves an HTML file and also shows the map here. Hover over a subdivision.

In [ ]:
from election_map.mapmaker import make_map
m = make_map(gdf, color='a_minus_b', compare=('Chow Olivia', 'Bradford Brad'), out='mayor_swing.html', title='Mayor: Chow vs Bradford')
m